# Resolving Fraudulent Entities in Peer-to-Peer Networks

P2P fraud losses are skyrocketing. In 2023, 8% of banking customers reported being victims of P2P scams. Identifying malicious actors is crucial to protecting customers.

Starting with anonymized financial transaction data, we will detect communities, identify important financial nodes, and resolve entities–providing impactful results with less effort than traditional analysis
## Getting the Data

This notebook expects the P2P graph to already be loaded into a Neo4j AuraDB instance. To build one
from synthetic data, run `01_generate_p2p_synthetic.ipynb` and import the resulting lakehouse tables
with the Neo4j workload in Fabric. The README describes the graph model this notebook expects.

## Set Up

First we need to install the graphdatascience package:

In [ ]:
%pip install graphdatascience==1.22


And load them:

In [ ]:
from graphdatascience.session import GdsSessions, AuraAPICredentials, DbmsConnectionInfo, AlgorithmCategory
from datetime import timedelta
import pandas as pd
import os


Finally, our secrets:

In [ ]:
import json
with open("/lakehouse/default/Files/secrets.json") as f:
    secrets = json.load(f)

In [ ]:
NEO4J_URI = secrets["NEO4J_URI"]
NEO4J_USERNAME = secrets["NEO4J_USERNAME"]
NEO4J_PASSWORD = secrets["NEO4J_PASSWORD"]
CLIENT_ID = secrets["CLIENT_ID"]
CLIENT_SECRET = secrets["CLIENT_SECRET"]
TENANT_ID = secrets["TENANT_ID"]

## Establishing a Session

We then use our secrets to establish a connection to our AuraDB

In [ ]:
sessions = GdsSessions(api_credentials=AuraAPICredentials(CLIENT_ID, CLIENT_SECRET, TENANT_ID))

# A session is pinned to the database it was created against, so a shared name
# collides (409) the moment you point at a new instance - including one whose
# original database has since been deleted.
DB_ID = NEO4J_URI.split("//")[1].split(".")[0]
name = f"fraud-{DB_ID}"

for s in sessions.list():
    s_name = getattr(s, "name", None)
    if s_name and s_name.startswith("fraud") and s_name != name:
        print(f"deleting orphaned session '{s_name}'")
        try:
            sessions.delete(session_name=s_name)
        except Exception as e:
            print(f"  could not delete: {type(e).__name__} - remove it in the Aura console")

print(f"session name: {name}")

memory = sessions.estimate(
    node_count=475,
    relationship_count=800,
    algorithm_categories=[AlgorithmCategory.CENTRALITY, AlgorithmCategory.NODE_EMBEDDING],
)

db_connection_info = DbmsConnectionInfo(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)

# Create or retrieve a session
gds = sessions.get_or_create(
    session_name=name,
    memory=memory,
    db_connection=db_connection_info, # this is checking for a bolt server currently
    ttl=timedelta(hours=5),
)


## Exploratory Analysis and Cleaning our Data

First let's take a look at the node labels for our graph. We quickly notice that the most common type is an IP address.

In [ ]:
gds.run_cypher('''
    CALL apoc.meta.stats()
    YIELD labels
    UNWIND keys(labels) AS nodeLabel
    RETURN nodeLabel, labels[nodeLabel] AS nodeCount
''')

Next let's do the same for relationships. Unsurprisingly, the most common type is HAS_IP.

In [ ]:
gds.run_cypher('''
    CALL apoc.meta.stats()
    YIELD relTypesCount
    UNWIND keys(relTypesCount) AS relationshipType
    RETURN relationshipType, relTypesCount[relationshipType] AS relationshipCount
''')

Next we are going to examine how many of our nodes has the preassigned fraud label:

In [ ]:
gds.run_cypher('MATCH(u:User) RETURN u.fraudMoneyTransfer AS fraudMoneyTransfer, count(u) AS cnt')

Then we are going to assign the FlaggedUser label to the nodes that are suspected fraudsters:

In [ ]:
gds.run_cypher('MATCH(u:User) WHERE u.fraudMoneyTransfer=1 SET u:FlaggedUser RETURN count(u)')

Next we are going to find users who have a transaction between them and also share a credit card. We are going to create a new relationship between them called P2P_WITH_SHARED_CARD:

In [ ]:
gds.run_cypher('''
    MATCH (u1:User)-[r:P2P]->(u2)
    WITH u1, u2, count(r) AS cnt
    MATCH (u1)-[:HAS_CC]->(n)<-[:HAS_CC]-(u2)
    WITH u1, u2, count(DISTINCT n) AS cnt
    MERGE(u1)-[s:P2P_WITH_SHARED_CARD]->(u2)
    RETURN count(DISTINCT s) AS cnt
''')

Next we are going to create a SHARED_IDS based on a few different business rules:

In [ ]:
gds.run_cypher('''
MATCH (u1:User)-[:HAS_CC|USED]->(n)<-[:HAS_CC|USED]-(u2:User)
WHERE u1.guid < u2.guid
  AND COUNT { (n)<--() } <= 10
WITH DISTINCT u1, u2

MATCH (u1)-[:HAS_CC|USED|HAS_IP]->(m)<-[:HAS_CC|USED|HAS_IP]-(u2)
WITH u1, u2, count(DISTINCT m) AS shared_count
WHERE shared_count > 2

MERGE (u1)-[s:SHARED_IDS]-(u2)
RETURN count(DISTINCT s)
''')

## Creating a Projection and Running Algorithms

Next we create a graph projection using these two new relationships that we just created. We use an OPTIONAL match to ensure that we keep those singleton communities of users who do not have a shared id or credit card:

In [ ]:
query = """
CALL {
  MATCH (u1:User)
  OPTIONAL MATCH (u1)-[r:SHARED_IDS|P2P_WITH_SHARED_CARD]-(u2:User)
  WHERE u1.guid < u2.guid
  RETURN u1 AS source, u2 AS target, type(r) AS relType
}
RETURN gds.graph.project.remote(source, target, {
    sourceNodeLabels: labels(source),
    targetNodeLabels: labels(target),
    relationshipType: relType
});
"""

# Project the graph into GDS
gds.graph.drop("full")
full, result = gds.graph.project(
    graph_name="full",
    query=query
)

### Weakly Connected Components

Weakly Connected Components (WCC) is a practical and highly scalable community detection algorithm. It is also deterministic and very explainable. It defines a community simply as a set of nodes connected by a subset of relationship types in the graph. This makes WCC a good choice for formal community assignment in production fraud detection settings.

In [ ]:
df = gds.wcc.write(full, writeProperty='wccId')

As these communities are meant to label underlying groups of individuals, if even one flagged account is in the community, we will label all user accounts in the group as fraud risks:

In [ ]:
gds.run_cypher('''
    MATCH (f:FlaggedUser)
    WITH collect(DISTINCT f.wccId) AS flaggedCommunities
    MATCH(u:User) WHERE u.wccId IN flaggedCommunities
    SET u:FraudRiskUser
    SET u.fraudRisk=1
    RETURN count(u)
''')


This gives us a total of 452 fraud risk accounts which means if we subtract the 241 already flagged accounts we identified 211 new fraud risk user accounts.

## Wrapping Things Up

The breakdown of communities by size is listed below. The majority are single user communities. Only a small portion have multiple users and of those, community sizes are mostly 2 and 3. Larger communities are rare. However, if we look at the fraudUser accounts we will see that the majority reside in multi-user communities. The 118 fraud accounts in single user communities are flagged users (via original chargeback logic) that have yet to be resolved to a community.

In [ ]:
gds.run_cypher( '''
    MATCH (u:User)
    WITH u.wccId AS community, count(u) AS cSize, sum(u.fraudRisk) AS cFraudSize
    WITH community, cSize, cFraudSize,
    CASE
        WHEN cSize=1 THEN ' 1'
        WHEN cSize=2 THEN ' 2'
        WHEN cSize=3 THEN ' 3'
        WHEN cSize>3 AND cSize<=10 THEN ' 4-10'
        WHEN cSize>10 AND cSize<=50 THEN '11-50'
        WHEN cSize>10 THEN '>50' END AS componentSize
    RETURN componentSize,
        count(*) AS numberOfComponents,
        sum(cSize) AS totalUserCount,
        sum(cFraudSize) AS fraudUserCount
    ORDER BY componentSize
''')

Fraud Risk labeling helped identify an additional 211 new fraud risk user accounts, nearly doubling the number of known fraud users (87.5 percent increase).

## Writing results to OneLake

Three Delta tables in `dbo`, separate from the source tables in `p2p_network`.
`fraud_accounts` is the investigator worklist, `fraud_groups` is one row per group,
`group_size_summary` is the distribution.

In [ ]:
USER_KEY   = "guid"

# Source tables live in p2p_network. Results go to dbo so the two stay separate.
SCHEMA     = "dbo"

# Fresh names. Repeated mode("overwrite") on the same table left the SQL analytics
# endpoint resolving Delta versions whose parquet files no longer exist; a
# never-before-used name has no stale catalog entry to reconcile.
T_USERS    = f"{SCHEMA}.fraud_accounts"
T_GROUPS   = f"{SCHEMA}.fraud_groups"
T_SIZES    = f"{SCHEMA}.group_size_summary"

chk = gds.run_cypher(f"""
MATCH (u:User)
RETURN count(u)            AS users,
       count(u.{USER_KEY}) AS with_key,
       count(u.wccId)      AS with_wcc
""").iloc[0]
print(chk.to_string())
assert chk["with_key"] == chk["users"], f"{USER_KEY} missing on some users"
assert chk["with_wcc"] == chk["users"], "run the WCC write cell first"

In [ ]:
cur = spark.sql("SELECT current_catalog() AS c, current_schema() AS s").collect()[0]
print(f"catalog {cur['c']} / schema {cur['s']}")

STALE = [f"{SCHEMA}.fraud_risk_users", f"{SCHEMA}.fraud_rings",
         f"{SCHEMA}.community_size_summary"]
for t in STALE + [T_USERS, T_GROUPS, T_SIZES]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("dropped:", ", ".join(STALE + [T_USERS, T_GROUPS, T_SIZES]))

### The worklist — every account sharing a group with a confirmed account

In [ ]:
users = gds.run_cypher(f"""
MATCH (u:User)
WITH u.wccId AS gid, collect(u) AS members
WITH gid, members,
     size(members) AS group_size,
     size([m IN members WHERE coalesce(m.fraudMoneyTransfer, 0) = 1]) AS group_confirmed_accounts
WHERE group_confirmed_accounts > 0
UNWIND members AS u
RETURN u.{USER_KEY}                              AS user_guid,
       gid                                       AS group_id,
       group_size,
       group_confirmed_accounts,
       coalesce(u.fraudMoneyTransfer, 0)         AS fraudMoneyTransfer,
       CASE WHEN coalesce(u.fraudMoneyTransfer, 0) = 1
            THEN 'confirmed' ELSE 'associated' END AS status,
       COUNT {{ (u)-[:SHARED_IDS|P2P_WITH_SHARED_CARD]-(c:User)
                WHERE c.fraudMoneyTransfer = 1 }} AS fraud_accounts_linked,
       COUNT {{
           MATCH (u)-[:HAS_CC|USED|HAS_IP]->(n)<-[:HAS_CC|USED|HAS_IP]-(f:User)
           WHERE f.fraudMoneyTransfer = 1 AND COUNT {{ (n)<--() }} <= 10
       }}                                        AS shared_identifiers_with_fraud,
       COUNT {{ (u)-[:P2P]-(:User) }}             AS p2p_counterparties,
       COUNT {{ (u)-[:SHARED_IDS]-(:User) }}      AS shared_id_links,
       COUNT {{ (u)-[:P2P_WITH_SHARED_CARD]-(:User) }} AS shared_card_links
ORDER BY shared_identifiers_with_fraud DESC, fraud_accounts_linked DESC
""")
print(f"{len(users):,} rows")
print(users["status"].value_counts().to_string())

### One row per group

In [ ]:
groups = gds.run_cypher(f"""
MATCH (u:User)
WITH u.wccId AS gid,
     count(u)                               AS group_size,
     sum(coalesce(u.fraudMoneyTransfer, 0)) AS group_confirmed_accounts,
     sum(coalesce(u.fraudRisk, 0))          AS at_risk,
     collect(u.{USER_KEY})                  AS members
WHERE group_confirmed_accounts > 0
RETURN gid AS group_id, group_size, group_confirmed_accounts,
       at_risk - group_confirmed_accounts                    AS group_associated_accounts,
       round(1.0 * group_confirmed_accounts / group_size, 3) AS group_confirmed_share,
       apoc.text.join(members[0..20], ', ')                  AS sample_members
ORDER BY group_size DESC, group_confirmed_accounts DESC
""")
print(f"{len(groups):,} groups")
print(groups.head(10).to_string(index=False))

### Size distribution

In [ ]:
sizes = gds.run_cypher("""
MATCH (u:User)
WITH u.wccId AS gid, count(u) AS n, sum(coalesce(u.fraudRisk, 0)) AS nf
WITH gid, n, nf,
     CASE WHEN n = 1 THEN '1' WHEN n = 2 THEN '2' WHEN n = 3 THEN '3'
          WHEN n <= 10 THEN '4-10' WHEN n <= 50 THEN '11-50'
          ELSE '>50' END AS group_size_band
RETURN group_size_band,
       count(*) AS number_of_groups,
       sum(n)   AS total_accounts,
       sum(nf)  AS accounts_in_flagged_groups
ORDER BY group_size_band
""")
print(sizes.to_string(index=False))

### Write

Attach the lakehouse and make it the **default** before running. `createDataFrame`
rejects object columns holding nulls, so types are coerced first.

In [ ]:
def to_delta(df, table, ints=(), floats=()):
    out = df.copy()
    for c in ints:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype("int64")
    for c in floats:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).astype("float64")
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = out[c].fillna("").astype(str)
    sdf = spark.createDataFrame(out)
    (sdf.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(table))
    loc = spark.sql(f"DESCRIBE DETAIL {table}").select("location").collect()[0][0]
    print(f"{table:22s} {sdf.count():>7,} rows  |  {loc}")

to_delta(users, T_USERS,
         ints=("group_id", "group_size", "group_confirmed_accounts",
               "fraudMoneyTransfer", "fraud_accounts_linked",
               "shared_identifiers_with_fraud", "p2p_counterparties",
               "shared_id_links", "shared_card_links"))

to_delta(groups, T_GROUPS,
         ints=("group_id", "group_size", "group_confirmed_accounts",
               "group_associated_accounts"),
         floats=("group_confirmed_share",))

to_delta(sizes, T_SIZES,
         ints=("number_of_groups", "total_accounts", "accounts_in_flagged_groups"))

### Verify

In [ ]:
n_graph = int(gds.run_cypher(
    "MATCH (u:User) WHERE u.fraudRisk = 1 RETURN count(u) AS n")["n"].iloc[0])
n_delta = spark.table(T_USERS).count()
print(f"graph fraudRisk=1 : {n_graph:,}")
print(f"{T_USERS:18s}: {n_delta:,}")
assert n_graph == n_delta, "row count drifted between AuraDB and OneLake"
spark.table(T_USERS).printSchema()
print("\nmatch")

## After writing — before touching the data agent

The SQL analytics endpoint has its own catalog and lags behind Spark. Open the
lakehouse, switch the view selector to **SQL analytics endpoint**, and run:

```sql
SELECT TOP 5 * FROM dbo.fraud_accounts;
```

If that fails with `underlying location does not exist`, the endpoint has not
synced. Hit **Refresh** in the Explorer toolbar and wait. Do **not** point the data
agent at the tables until this query works — the agent reads through this same
endpoint, so no amount of recreating the agent will help.

## Grounding text for the data agent

> `fraud_accounts` holds one row per account that shares a resolved-identity group
> with at least one confirmed fraudulent account. Groups come from weakly connected
> components over two derived relationships: `SHARED_IDS` (accounts sharing more
> than two of the same card, device or IP, where the shared identifier is used by
> ten accounts or fewer) and `P2P_WITH_SHARED_CARD` (accounts that both transacted
> with each other and share a credit card).
>
> `fraudMoneyTransfer` is 1 for confirmed fraud and 0 otherwise.
> `shared_identifiers_with_fraud` counts the cards, devices and IPs this account
> shares with confirmed accounts, ignoring identifiers used by more than ten
> accounts. `fraud_accounts_linked` counts the confirmed accounts it is directly
> linked to. Columns prefixed `group_` describe the group, not the account.
>
> `fraud_groups` has one row per group; `sample_members` lists up to twenty guids.

## Troubleshooting

| Symptom | Fix |
|---|---|
| `underlying location does not exist` from the SQL endpoint | repeated `overwrite` left the endpoint on a dead Delta version. Refresh the endpoint; if still failing after ~30 min, write to a table name never used before |
| tables visible in Spark but not to the agent | the agent reads the SQL analytics endpoint, not the Delta log. Verify with T-SQL first |
| agent filters on a column that no longer exists | endpoint schema cache; refresh the endpoint, then re-add the data source |
| `assert` on `with_wcc` | the WCC write cell has not run, or the session was recycled before write-back finished |
| `Livy session has failed` | the Fabric Spark session was recycled (idle timeout), unrelated to the GDS session |
| `AuraApiError 409 ... different parameters` | a GDS session of that name is pinned to another database; the session cell now names it after the database |
| `createDataFrame` type error | an object column still holds nulls; add it to `ints` or `floats` in `to_delta` |
| `apoc.text.join` unknown | drop `sample_members` from the groups query |
| projection returns 0 nodes | `p.orderCount`-style property unset, or `fraudMoneyTransfer` null — `null >= N` filters everything out |

Finally, we delete our session:

In [ ]:
sessions.delete(session_name=name)   # name is set in the session cell